In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Feature Engineering + Preprocessing Pipeline

In [2]:
df = pd.read_csv("../data/processed/cleaned_data.csv")

### Task 1 — Identify Feature Types

Categorizing all 37 columns into three groups:

- binary (0/1) → no scaling
- ordinal/continuous → needs scaling
- target → separate, never scale

In [ ]:
binary_cols = ["_RFBING5",'_AIDTST3','HLTHPLN1','QLACTLM2','CHCOCNCR',
               '_RFHLTH','HAVARTH3','HIVTST6','MEDCOST','CHCSCNCR',
               'EXERANY2','DRNKANY5','_HCVU651','ADDEPEV2','PERSDOC2',
               'CVDSTRK3','_TOTINDA','CHCKIDNY','CVDCRHD4','CVDINFR4',
               'SEX','_DRDXAR1']  # PNEUVAC3 removed
continuous_cols = ["PHYSHLTH", "_BMI5", "ALCDAY5", "MENTHLTH"]
ordinal_cols = ["_SMOKER3", "GENHLTH", "_AGEG5YR", "INCOME2", "EDUCA", "_ASTHMS1", "CHECKUP1", "_CHLDCNT"]
target = ["Diabetes_binary"]

In [ ]:
print(ordinal_cols)
print(continuous_cols)

['_SMOKER3', 'GENHLTH', 'MENTHLTH', '_AGEG5YR', 'INCOME2', 'EDUCA', '_ASTHMS1', 'CHECKUP1', '_CHLDCNT']
['PHYSHLTH', '_BMI5', 'ALCDAY5']


### Task 2 — Train/Test Split + Preprocessing Pipeline

**Goal:** Split the dataset for unbiased evaluation and build a preprocessing pipeline that respects feature types.

**Plan:**
- Create $X$ and $y$, then split into train/test using stratification to preserve class imbalance.
- Build a `ColumnTransformer` with type-specific steps: passthrough for binary, scaling for continuous, and ordinal encoding for ordered categories.
- Fit the preprocessor on training data only, then transform the test set to avoid leakage.

In [7]:
x = df.drop(columns=target)
y = df["Diabetes_binary"]
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [9]:
preprocessor = ColumnTransformer(transformers=[
    ("binary","passthrough",binary_cols),
    ("continous",StandardScaler(),continuous_cols),
    ("ordinal",OrdinalEncoder(),ordinal_cols)
])

In [10]:
x_train = preprocessor.fit_transform(x_train)
x_test = preprocessor.transform(x_test)

In [ ]:
all_cols = binary_cols + continuous_cols + ordinal_cols
x_train = pd.DataFrame(x_train,columns=all_cols)
x_test = pd.DataFrame(x_test,columns=all_cols)

In [64]:
print(x_train.shape)
print(x_test.shape)
print(x_train.head())

(1614856, 35)
(403715, 35)
   _RFBING5  _AIDTST3  HLTHPLN1  QLACTLM2  CHCOCNCR  _RFHLTH  HAVARTH3  \
0       0.0       0.0       1.0       1.0       0.0      0.0       1.0   
1       0.0       1.0       1.0       1.0       0.0      1.0       0.0   
2       0.0       1.0       1.0       0.0       0.0      1.0       1.0   
3       0.0       0.0       1.0       0.0       0.0      1.0       0.0   
4       0.0       1.0       1.0       0.0       0.0      1.0       0.0   

   HIVTST6  MEDCOST  CHCSCNCR  ...   ALCDAY5  _SMOKER3  GENHLTH  MENTHLTH  \
0      0.0      0.0       0.0  ...  1.168664       2.0      3.0       0.0   
1      1.0      0.0       0.0  ...  1.158154       3.0      1.0       5.0   
2      1.0      0.0       0.0  ...  1.179174       3.0      1.0       0.0   
3      0.0      0.0       0.0  ... -0.954354       3.0      1.0       0.0   
4      1.0      0.0       0.0  ... -0.954354       3.0      0.0       0.0   

   _AGEG5YR  INCOME2  EDUCA  _ASTHMS1  CHECKUP1  _CHLDCNT  
0    

In [65]:
print(x_train["INCOME2"].value_counts())

INCOME2
7.0    622738
6.0    229040
5.0    209135
4.0    159733
3.0    132518
2.0    107271
1.0     80411
0.0     74010
Name: count, dtype: int64


In [66]:
x_train.to_csv("../data/processed/x_train.csv", index=False)
x_test.to_csv("../data/processed/x_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [11]:
import joblib
joblib.dump(preprocessor, '../models/preprocessor.pkl')
print("Preprocessor saved.")

Preprocessor saved.


## Documentation for Preprocessing (Day 8)

1) Train-test split
    - Used `train_test_split` with test_size=0.2, random_state=42, and stratify=y to preserve the 84/16 class imbalance
    - Splitting before preprocessing keeps evaluation unbiased and mirrors real-world inference

2) Preprocessing strategy
    - Binary features: passthrough (no scaling required)
    - Continuous features: standardized using StandardScaler to normalize scale
    - Ordinal features: encoded using OrdinalEncoder to preserve category order

3) Leakage prevention
    - Fit the preprocessing pipeline only on the training data
    - Applied the exact same transform to the test set

4) Saved artifacts
    - x_train, x_test, y_train, y_test saved to data/processed for Week 4 model training